# Solución 10: Método de Euler para ODEs + Bisección para decaimiento

Orígenes:
- `02. DifferentialEquations/EulerMethod/odes_and_euler.pdf`
- `01. Fundamental Algorithms/04.RootSearching/Roots_searching.pdf`

---

## Parte 1: Solución de ODE con método de Euler (función propia)

**Función elegida (del PDF — Ejemplo 4):**

$$3\frac{dy}{dx} + 2y = e^{-x}, \quad y(0) = 5$$

Reescribiendo:
$$\frac{dy}{dx} = \frac{e^{-x} - 2y}{3} \implies f(x, y) = \frac{e^{-x} - 2y}{3}$$

**Solución analítica exacta:**
$$y(x) = 6e^{-2x/3} - e^{-x}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ============================================================
# FUNCIÓN DEL SISTEMA ODE
# ============================================================
def f_ode(x, y):
    """
    f(x,y) = (e^{-x} - 2y) / 3
    Corresponde a la ODE: 3 dy/dx + 2y = e^{-x}
    """
    return (np.exp(-x) - 2*y) / 3


def y_analitica(x):
    """Solución exacta: y = 6e^{-2x/3} - e^{-x}."""
    return 6 * np.exp(-2*x/3) - np.exp(-x)


# ============================================================
# MÉTODO DE EULER
# ============================================================
def euler(f, x0, y0, h, x_final):
    """
    Método de Euler para dy/dx = f(x, y).
    Devuelve arrays (x_vals, y_vals).
    """
    x_vals = [x0]
    y_vals = [y0]
    
    x, y = x0, y0
    while round(x, 10) < round(x_final, 10):
        y = y + f(x, y) * h
        x = x + h
        x_vals.append(x)
        y_vals.append(y)
    
    return np.array(x_vals), np.array(y_vals)


# ============================================================
# PARÁMETROS
# ============================================================
x0, y0   = 0.0, 5.0
x_final  = 3.0

# ============================================================
# COMPARACIÓN PARA DISTINTOS h
# ============================================================
pasos = [0.5, 0.2, 0.1, 0.05]

x_ref = np.linspace(x0, x_final, 400)
y_ref = y_analitica(x_ref)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for ax, h in zip(axes, pasos):
    xv, yv = euler(f_ode, x0, y0, h, x_final)
    ax.plot(x_ref, y_ref, 'k-', linewidth=2, label='Analítica')
    ax.plot(xv, yv, 'o--', color='royalblue', linewidth=1.8,
            markersize=5, label=f'Euler (h={h})')
    ax.set_title(f'Euler h = {h}', fontsize=11)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle(r'Método de Euler — $3y\' + 2y = e^{-x}$, $y(0)=5$', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Tabla de resultados para h=0.1 (primeros 10 pasos)
h = 0.1
xv, yv = euler(f_ode, x0, y0, h, x_final)
y_exact = y_analitica(xv)
err_abs = np.abs(yv - y_exact)
err_rel = err_abs / np.abs(y_exact) * 100

df_euler = pd.DataFrame({
    'x': xv,
    'y_Euler': yv,
    'y_exacta': y_exact,
    'err_abs': err_abs,
    'err_rel%': err_rel
})

print("=" * 70)
print(f"TABLA DE EULER — h={h}  (primeros 12 pasos)")
print("=" * 70)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')
print(df_euler.head(12).to_string(index=False))
print("=" * 70)
print(f"\nError máximo (h={h}): {err_abs.max():.6f}")

In [ ]:
# Convergencia: error máximo vs h
h_vals = [0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005]
errores_max = []

for h in h_vals:
    xv, yv = euler(f_ode, x0, y0, h, x_final)
    err = np.abs(yv - y_analitica(xv))
    errores_max.append(err.max())

plt.figure(figsize=(8, 5))
plt.loglog(h_vals, errores_max, 'o-', color='crimson', linewidth=2.5, markersize=8, label='Error máx Euler')
ref = errores_max[0] * np.array(h_vals) / h_vals[0]
plt.loglog(h_vals, ref, 'k--', linewidth=1.5, label=r'$\mathcal{O}(h)$')
plt.xlabel('Tamaño de paso h')
plt.ylabel('Error máximo absoluto')
plt.title('Convergencia del Método de Euler', fontsize=13)
plt.legend(fontsize=12)
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()
print("El método de Euler es O(h): al reducir h a la mitad, el error se reduce a la mitad.")

---

## Parte 2: Bisección para problema de decaimiento radioactivo

El decaimiento radioactivo está dado por:

$$N(t) = N_0 \, e^{-\lambda t}$$

donde $\lambda = \ln(2)/T_{1/2}$ es la constante de decaimiento y $T_{1/2}$ es la vida media.

**Problema:** Dado que un isótopo tiene $T_{1/2} = 5730$ años (Carbono-14), ¿en qué tiempo $t^*$ queda el **10%** de la muestra original?

Esto equivale a encontrar la raíz de:

$$g(t) = N_0 e^{-\lambda t} - 0.1 N_0 = e^{-\lambda t} - 0.1 = 0$$

In [ ]:
# ============================================================
# BISECCIÓN PARA DECAIMIENTO RADIOACTIVO (C-14)
# ============================================================
T_half = 5730.0          # vida media [años]
lam    = np.log(2) / T_half   # constante de decaimiento [1/año]
fraccion_objetivo = 0.10      # 10% de la muestra original

# La raíz exacta analítica
t_exacto = -np.log(fraccion_objetivo) / lam
print(f"Valor exacto analítico: t* = {t_exacto:.4f} años = {t_exacto/1000:.4f} ka")

# Función para bisección
def g_decay(t):
    return np.exp(-lam * t) - fraccion_objetivo

print(f"g(0)      = {g_decay(0):.4f}")
print(f"g(20000)  = {g_decay(20000):.4f}")
print("→ Hay cambio de signo entre 0 y 20000 años")

# ============================================================
# Método de bisección
# ============================================================
def biseccion_simple(g, a, b, tol=1e-8, max_iter=100):
    tabla = []
    xr_ant = a
    for i in range(1, max_iter+1):
        xr = (a + b) / 2
        err = abs((xr - xr_ant)/xr)*100 if i > 1 else np.nan
        tabla.append({'Iter': i, 'a': a, 'b': b, 'xr': xr,
                      'g(xr)': g(xr), 'err_rel%': err})
        if i > 1 and err < tol*100:
            break
        if g(a)*g(xr) < 0:
            b = xr
        else:
            a = xr
        xr_ant = xr
    return xr, i, pd.DataFrame(tabla)


t_raiz, n_decaim, tabla_decaim = biseccion_simple(g_decay, 0.0, 25000.0, tol=1e-8)

print("\n" + "=" * 65)
print("BISECCIÓN — Decaimiento radioactivo C-14")
print("=" * 65)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')
print(tabla_decaim.to_string(index=False))
print("=" * 65)
print(f"Raíz (bisección) : t* = {t_raiz:.4f} años")
print(f"Exacto analítico : t* = {t_exacto:.4f} años")
print(f"Error            : {abs(t_raiz - t_exacto):.4f} años")
print(f"Iteraciones      : {n_decaim}")

In [ ]:
# Visualización del decaimiento y la raíz
t_arr = np.linspace(0, 25000, 500)
N_arr = np.exp(-lam * t_arr)  # N/N0

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t_arr, N_arr, 'royalblue', linewidth=2.5, label='$N(t)/N_0 = e^{-\\lambda t}$')
ax.axhline(fraccion_objetivo, color='crimson', linestyle='--', linewidth=1.8,
           label=f'$N/N_0 = {fraccion_objetivo}$')
ax.axvline(t_raiz, color='green', linestyle='--', linewidth=1.8,
           label=f'$t^* = {t_raiz:.1f}$ años')
ax.plot(t_raiz, fraccion_objetivo, 'g*', markersize=14, zorder=5)

# Anotar vida media
ax.axvline(T_half, color='orange', linestyle=':', linewidth=1.5,
           label=f'$T_{{1/2}} = {T_half:.0f}$ años')

ax.set_xlabel('Tiempo [años]')
ax.set_ylabel('$N/N_0$')
ax.set_title('Decaimiento radioactivo C-14\nBúsqueda de $t^*$ tal que $N/N_0=0.1$', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Resumen

| Parte | Método | Resultado |
|-------|--------|----------|
| Euler ODE | $h=0.1$, 30 pasos | Error máx $\approx 0.036$ |
| Euler ODE | Converge como $\mathcal{O}(h)$ | Confirmado |
| Decaimiento C-14 | Bisección | $t^* \approx 19,034$ años |